# DeepSeismic2 — Data Exploration

This notebook walks through loading, visualizing, and converting seismic data
from the Volve dataset (or synthetic sample). It demonstrates:

1. Loading SEG-Y files with segyio
2. Examining survey geometry and trace headers
3. Visualizing seismic sections (inline, crossline, time slices)
4. Amplitude statistics and QC
5. Converting to Zarr format for the ML pipeline

**Prerequisites:** Run `python scripts/download_volve.py --sample` first to generate test data.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import segyio

# Add project root to path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root / 'src'))

# Data paths
DATA_DIR = project_root / 'data' / 'volve'
SEISMIC_DIR = DATA_DIR / 'seismic'

# Use sample data or real Volve data
segy_file = SEISMIC_DIR / 'sample_volume.segy'
if not segy_file.exists():
    # Try the real Volve file
    segy_file = SEISMIC_DIR / 'ST10010ZC11_PZ_PSDM_KIRCH_FAR_D.MIG_FIN.POST_STACK.3D.JS-017536.segy'

if not segy_file.exists():
    print('⚠️  No SEG-Y data found!')
    print('    Run: python scripts/download_volve.py --sample')
    print(f'    Expected at: {segy_file}')
else:
    print(f'✅ Using: {segy_file.name} ({segy_file.stat().st_size / 1024 / 1024:.1f} MB)')

## 1. Load SEG-Y and Inspect Geometry

In [ ]:
with segyio.open(str(segy_file), 'r', strict=False) as f:
    print('=== Survey Geometry ===')
    print(f'  Format:     {f.format}')
    print(f'  Traces:     {f.tracecount:,}')
    print(f'  Samples:    {len(f.samples)}')
    print(f'  Sample rate: {f.samples[1] - f.samples[0]:.1f} ms')
    print(f'  Time range: {f.samples[0]:.0f} – {f.samples[-1]:.0f} ms')
    print()
    print(f'  Inlines:    {f.ilines[0]} – {f.ilines[-1]} ({len(f.ilines)} total)')
    print(f'  Crosslines: {f.xlines[0]} – {f.xlines[-1]} ({len(f.xlines)} total)')
    print()
    
    # Read a sample of trace headers for coordinate info
    cdp_x = f.attributes(segyio.TraceField.CDP_X)[:10]
    cdp_y = f.attributes(segyio.TraceField.CDP_Y)[:10]
    print(f'  CDP X range (first 10): {min(cdp_x)} – {max(cdp_x)}')
    print(f'  CDP Y range (first 10): {min(cdp_y)} – {max(cdp_y)}')

## 2. Amplitude Statistics

In [ ]:
with segyio.open(str(segy_file), 'r', strict=False) as f:
    # Read all traces into memory (fine for sample data / small volumes)
    # For large real data, sample every Nth trace
    n_traces = f.tracecount
    stride = max(1, n_traces // 1000)  # Sample ~1000 traces max
    
    amplitudes = []
    for i in range(0, n_traces, stride):
        amplitudes.append(f.trace[i])
    amplitudes = np.array(amplitudes)

print(f'Sampled {len(amplitudes)} traces (stride={stride})')
print(f'\n=== Amplitude Statistics ===')
print(f'  Min:    {amplitudes.min():.6f}')
print(f'  Max:    {amplitudes.max():.6f}')
print(f'  Mean:   {amplitudes.mean():.6f}')
print(f'  Std:    {amplitudes.std():.6f}')
print(f'  RMS:    {np.sqrt(np.mean(amplitudes**2)):.6f}')
print(f'  Zeros:  {(amplitudes == 0).sum()} ({100*(amplitudes==0).sum()/amplitudes.size:.1f}%)')

In [ ]:
try:
    import matplotlib.pyplot as plt
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    # Amplitude histogram
    axes[0].hist(amplitudes.ravel(), bins=100, color='steelblue', alpha=0.7, log=True)
    axes[0].set_xlabel('Amplitude')
    axes[0].set_ylabel('Count (log)')
    axes[0].set_title('Amplitude Distribution')
    axes[0].axvline(0, color='red', linestyle='--', alpha=0.5)
    
    # Trace display (first 50 traces)
    axes[1].imshow(amplitudes[:50].T, aspect='auto', cmap='seismic',
                   vmin=-np.percentile(np.abs(amplitudes), 95),
                   vmax=np.percentile(np.abs(amplitudes), 95))
    axes[1].set_xlabel('Trace Number')
    axes[1].set_ylabel('Sample')
    axes[1].set_title('First 50 Traces (Wiggle Area)')
    
    plt.tight_layout()
    plt.show()
except ImportError:
    print('matplotlib not installed — skipping plots')
    print('Install with: pip install matplotlib')

## 3. Seismic Section Display

In [ ]:
with segyio.open(str(segy_file), 'r', strict=False) as f:
    # Read a single inline section
    mid_inline = f.ilines[len(f.ilines) // 2]
    inline_section = f.iline[mid_inline]  # shape: (n_xlines, n_samples)
    
    # Read a crossline section
    mid_xline = f.xlines[len(f.xlines) // 2]
    xline_section = f.xline[mid_xline]  # shape: (n_ilines, n_samples)
    
    samples = f.samples
    xlines = f.xlines
    ilines = f.ilines

print(f'Inline {mid_inline}: {inline_section.shape}')
print(f'Crossline {mid_xline}: {xline_section.shape}')

In [ ]:
try:
    import matplotlib.pyplot as plt
    
    clip = np.percentile(np.abs(inline_section), 95)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    
    # Inline section
    im1 = axes[0].imshow(inline_section.T, aspect='auto', cmap='seismic',
                         vmin=-clip, vmax=clip,
                         extent=[xlines[0], xlines[-1], samples[-1], samples[0]])
    axes[0].set_xlabel('Crossline')
    axes[0].set_ylabel('Time (ms)')
    axes[0].set_title(f'Inline {mid_inline}')
    plt.colorbar(im1, ax=axes[0], label='Amplitude')
    
    # Crossline section
    im2 = axes[1].imshow(xline_section.T, aspect='auto', cmap='seismic',
                         vmin=-clip, vmax=clip,
                         extent=[ilines[0], ilines[-1], samples[-1], samples[0]])
    axes[1].set_xlabel('Inline')
    axes[1].set_ylabel('Time (ms)')
    axes[1].set_title(f'Crossline {mid_xline}')
    plt.colorbar(im2, ax=axes[1], label='Amplitude')
    
    plt.suptitle('Volve Seismic Sections', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
except ImportError:
    print('matplotlib not installed — skipping plots')

## 4. Time Slice

In [ ]:
with segyio.open(str(segy_file), 'r', strict=False) as f:
    # Build a 3D volume (for sample data this fits in memory)
    n_il = len(f.ilines)
    n_xl = len(f.xlines)
    n_s = len(f.samples)
    
    # Only load full volume if small enough (<500MB)
    estimated_size = n_il * n_xl * n_s * 4 / (1024**2)
    print(f'Volume size: {n_il}×{n_xl}×{n_s} = {estimated_size:.0f} MB')
    
    if estimated_size < 500:
        volume = np.zeros((n_il, n_xl, n_s), dtype=np.float32)
        for i, il in enumerate(f.ilines):
            volume[i] = f.iline[il]
        print(f'✅ Full volume loaded: {volume.shape}')
    else:
        print('⚠️  Volume too large for memory — sampling 10 inlines')
        sample_ils = f.ilines[::n_il//10]
        volume = np.zeros((len(sample_ils), n_xl, n_s), dtype=np.float32)
        for i, il in enumerate(sample_ils):
            volume[i] = f.iline[il]

In [ ]:
try:
    import matplotlib.pyplot as plt
    
    # Time slice at ~middle of the volume
    time_idx = n_s // 2
    time_ms = samples[time_idx]
    time_slice = volume[:, :, time_idx]
    
    clip = np.percentile(np.abs(time_slice), 95)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(time_slice.T, aspect='auto', cmap='seismic',
                   vmin=-clip, vmax=clip,
                   extent=[ilines[0], ilines[-1], xlines[-1], xlines[0]])
    ax.set_xlabel('Inline')
    ax.set_ylabel('Crossline')
    ax.set_title(f'Time Slice at {time_ms:.0f} ms')
    plt.colorbar(im, label='Amplitude')
    plt.tight_layout()
    plt.show()
    
except ImportError:
    print('matplotlib not installed — skipping plots')

## 5. Convert to Zarr (Pipeline Integration)

In [ ]:
from deepseismic.ingest.segy_loader import segy_to_zarr, load_segy

# Convert SEG-Y to Zarr (our pipeline's working format)
zarr_path = DATA_DIR / 'zarr' / 'sample'
zarr_path.parent.mkdir(parents=True, exist_ok=True)

print('Converting SEG-Y → Zarr...')
metadata = segy_to_zarr(str(segy_file), str(zarr_path))
print(f'\n✅ Zarr store created at: {zarr_path}')
print(f'   Metadata: {metadata}')

In [ ]:
import zarr

# Verify the Zarr conversion
store = zarr.open(str(zarr_path), mode='r')
print('=== Zarr Store Contents ===')
print(f'  Type: {type(store)}')

# List arrays/groups
if hasattr(store, 'arrays'):
    for name, arr in store.arrays():
        print(f'  Array: {name} — shape={arr.shape}, dtype={arr.dtype}, chunks={arr.chunks}')
elif hasattr(store, 'shape'):
    print(f'  Root array — shape={store.shape}, dtype={store.dtype}')
    if hasattr(store, 'chunks'):
        print(f'  Chunks: {store.chunks}')

## 6. Next Steps

With data loaded and converted:

1. **Generate fault labels:** `from deepseismic.ingest.label_generator import generate_fault_mask`
2. **Extract patches:** `from deepseismic.preprocessing.patches import PatchDataset`
3. **Train UNet:** See `src/deepseismic/models/unet.py`
4. **Run inference:** `from deepseismic.models.inference import run_inference`
5. **Query via agent:** Start the API + agent for natural language interpretation

See `README.md` for full pipeline instructions.